In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import cross_validate, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import LinearSVC, SVC
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import make_scorer, f1_score, precision_score, recall_score
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import StackingClassifier
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import ConfusionMatrixDisplay
import optuna
from flaml import AutoML
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split

In [2]:
train = pd.read_csv('data/train_new_processed.csv')
test = pd.read_csv('data/test_new_processed.csv')

In [4]:
test.columns

Index(['Id', 'Elevation', 'Slope', 'Horizontal_Distance_To_Hydrology',
       'Vertical_Distance_To_Hydrology', 'Horizontal_Distance_To_Roadways',
       'Hillshade_9am', 'Hillshade_Noon', 'Hillshade_3pm',
       'Horizontal_Distance_To_Fire_Points', 'Wilderness_Area1',
       'Wilderness_Area2', 'Wilderness_Area3', 'Wilderness_Area4',
       'Soil_Type1', 'Soil_Type2', 'Soil_Type3', 'Soil_Type4', 'Soil_Type5',
       'Soil_Type6', 'Soil_Type7', 'Soil_Type8', 'Soil_Type9', 'Soil_Type10',
       'Soil_Type11', 'Soil_Type12', 'Soil_Type13', 'Soil_Type14',
       'Soil_Type16', 'Soil_Type17', 'Soil_Type18', 'Soil_Type19',
       'Soil_Type20', 'Soil_Type21', 'Soil_Type22', 'Soil_Type23',
       'Soil_Type24', 'Soil_Type25', 'Soil_Type26', 'Soil_Type27',
       'Soil_Type28', 'Soil_Type29', 'Soil_Type30', 'Soil_Type31',
       'Soil_Type32', 'Soil_Type33', 'Soil_Type34', 'Soil_Type35',
       'Soil_Type36', 'Soil_Type37', 'Soil_Type38', 'Soil_Type39',
       'Soil_Type40', 'Distance_To_Hyd

In [6]:
X = train.drop(columns=["Cover_Type", "Id"])
y = train["Cover_Type"]
y_shifted = y - 1

In [8]:
X_test = test.drop(columns=["Id"])

In [12]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

## Auto ML

In [7]:
automl = AutoML()

automl.fit(
    X_train=X,
    y_train=y_shifted,
    time_budget= 1800,  
    metric= "accuracy",
    task= "classification",
    seed= 42,
)

print(f"Best ML learner: {automl.best_estimator}")
print(f"Best hyperparameter config: {automl.best_config}")
print(f"Best accuracy on validation data: {1 - automl.best_loss}")

[flaml.automl.logger: 02-22 20:23:26] {2375} INFO - task = classification
[flaml.automl.logger: 02-22 20:23:26] {2386} INFO - Evaluation method: cv
[flaml.automl.logger: 02-22 20:23:26] {2489} INFO - Minimizing error metric: 1-accuracy
[flaml.automl.logger: 02-22 20:23:26] {2606} INFO - List of ML learners in AutoML Run: ['lgbm', 'rf', 'xgboost', 'extra_tree', 'xgb_limitdepth', 'sgd', 'lrl1']
[flaml.automl.logger: 02-22 20:23:26] {2911} INFO - iteration 0, current learner lgbm
[flaml.automl.logger: 02-22 20:23:27] {3046} INFO - Estimated sufficient time budget=3933s. Estimated necessary time budget=91s.
[flaml.automl.logger: 02-22 20:23:27] {3097} INFO -  at 0.5s,	estimator lgbm's best error=3.3651e-01,	best estimator lgbm's best error=3.3651e-01
[flaml.automl.logger: 02-22 20:23:27] {2911} INFO - iteration 1, current learner lgbm
[flaml.automl.logger: 02-22 20:23:27] {3097} INFO -  at 0.9s,	estimator lgbm's best error=3.3651e-01,	best estimator lgbm's best error=3.3651e-01
[flaml.auto

In [9]:
preds_0_indexed = automl.predict(X_test)
final_preds = preds_0_indexed + 1

submission = pd.DataFrame({
    "Id": test["Id"],
    "Cover_Type": final_preds
})

submission.to_csv("automl_1800_newfeatures_submission.csv", index=False)

## Stacking

In [10]:
base_models = [
    ("rf", RandomForestClassifier(random_state=42)),
    ("lgbm", LGBMClassifier(random_state=42)),
    ("xgb", XGBClassifier(random_state=42))
]

meta_learner = LogisticRegression(max_iter=1000)

stack_model = StackingClassifier(
    estimators=base_models,
    final_estimator=meta_learner,
    cv=5,
    n_jobs=-1,
    passthrough=False
)


stack_model.fit(X, y_shifted)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000460 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4375
[LightGBM] [Info] Number of data points in the train set: 15120, number of used features: 54
[LightGBM] [Info] Start training from score -1.945910
[LightGBM] [Info] Start training from score -1.945910
[LightGBM] [Info] Start training from score -1.945910
[LightGBM] [Info] Start training from score -1.945910
[LightGBM] [Info] Start training from score -1.945910
[LightGBM] [Info] Start training from score -1.945910
[LightGBM] [Info] Start training from score -1.945910
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.003814 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.003404 seconds.
You

,estimators,"[('rf', ...), ('lgbm', ...), ...]"
,final_estimator,LogisticRegre...max_iter=1000)
,cv,5
,stack_method,'auto'
,n_jobs,-1
,passthrough,False
,verbose,0
,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2


In [13]:
stack_cv_results = cross_validate(
    stack_model, X, y_shifted, 
    cv=skf, 
    scoring="accuracy", 
    n_jobs=-1
)

stack_mean_acc = stack_cv_results["test_score"].mean()
print(f"Stacked Ensemble Mean Accuracy: {stack_mean_acc:.4f}")

[LightGBM] [Info] [LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.003132 seconds.
You can set `force_col_wise=true` to remove the overhead.Auto-choosing row-wise multi-threading, the overhead of testing was 0.001156 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.

[LightGBM] [Info] Total Bins 4366
[LightGBM] [Info] Total Bins 4370
[LightGBM] [Info] Number of data points in the train set: 12096, number of used features: 54
[LightGBM] [Info] Number of data points in the train set: 12096, number of used features: 54
[LightGBM] [Info] Start training from score -1.945910
[LightGBM] [Info] Start training from score -1.945910
[LightGBM] [Info] Start training from score -1.945910
[LightGBM] [Info] Start training from score -1.945910
[LightGBM] [Info] Start training from score -1.945910
[LightGBM] [Info] Start training from score -1.945910
[LightGBM] [Info] Start training f

In [14]:
stack_preds = stack_model.predict(X_test)
final_stack_preds = stack_preds + 1

submission = pd.DataFrame({
    "Id": test["Id"],
    "Cover_Type": final_stack_preds
})
submission.to_csv("submission_stacked_newfeatuers_ensemble.csv", index=False)

## Super stack

In [ ]:
from catboost import CatBoostClassifier
from sklearn.ensemble import ExtraTreesClassifier

In [ ]:
base_models = [
    ('rf', RandomForestClassifier(n_estimators=300, max_depth=20, random_state=42)),
    ('et', ExtraTreesClassifier(n_estimators=300, random_state=42)),
    ('xgb', XGBClassifier(n_estimators=300, learning_rate=0.05, random_state=42)),
    ('lgbm', LGBMClassifier(n_estimators=300, learning_rate=0.05, random_state=42)),
    ('cat', CatBoostClassifier(iterations=300, silent=True, random_state=42)),
    ('knn', KNeighborsClassifier(n_neighbors=5)),
    ('mlp', MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=500, random_state=42))
]

meta_learner = LogisticRegression(max_iter=2000)

mega_stack = StackingClassifier(
    estimators=base_models,
    final_estimator=meta_learner,
    cv=5, 
    n_jobs=-1,
    passthrough=False
)

mega_stack.fit(X, y_shifted)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.005280 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4375
[LightGBM] [Info] Number of data points in the train set: 15120, number of used features: 54
[LightGBM] [Info] Start training from score -1.945910
[LightGBM] [Info] Start training from score -1.945910
[LightGBM] [Info] Start training from score -1.945910
[LightGBM] [Info] Start training from score -1.945910
[LightGBM] [Info] Start training from score -1.945910
[LightGBM] [Info] Start training from score -1.945910
[LightGBM] [Info] Start training from score -1.945910
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002123 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Auto-choosing col-wise mu

,estimators,"[('rf', ...), ('et', ...), ...]"
,final_estimator,LogisticRegre...max_iter=2000)
,cv,5
,stack_method,'auto'
,n_jobs,-1
,passthrough,True
,verbose,0
,n_estimators,300
,criterion,'gini'
,max_depth,20
,min_samples_split,2


In [27]:
m_stack_preds_shifted = mega_stack.predict(X_test)
m_stack_preds = m_stack_preds_shifted + 1

submission = pd.DataFrame({
    "Id": test["Id"],
    "Cover_Type": m_stack_preds
})
submission.to_csv("submission_morestackedtrueAndReg_nf.csv", index=False)